In [1]:
# minimal_unet_bias_training.py
import math, random
from dataclasses import dataclass
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [2]:
# --------------------------
# 1) Toy dataset (replace with real)
# --------------------------
class ToyBiasDataset(Dataset):
    """
    Yields (x, y) where:
      x: [C_in, H, W] state features (normalized)
      y: [C_out, H, W] target correction tendencies
    Replace the synthetic generator with your preprocessed data tensors.
    """
    def __init__(self, n=2000, c_in=8, c_out=2, h=64, w=128, seed=0):
        rng = np.random.default_rng(seed)
        self.x = rng.standard_normal((n, c_in, h, w)).astype(np.float32)

        # Create a smooth latent "truth" field and a biased "model" field
        def smooth_noise(nc):
            kx = rng.standard_normal((n, 1, h, w)).astype(np.float32)
            # cheap spectral smoothing
            KX = np.fft.rfft2(kx, axes=(-2, -1))
            ky = np.fft.rfftfreq(w)
            kx1 = np.fft.fftfreq(h)
            damp = 1.0 / (1.0 + (kx1[:, None]**2 + ky[None, :]**2)[None, None, :, :])
            KX *= damp
            sm = np.fft.irfft2(KX, s=(h, w), axes=(-2, -1)).astype(np.float32)
            return sm.repeat(nc, axis=1)

        truth = smooth_noise(c_out)
        model = truth + 0.5 * smooth_noise(c_out)  # add bias
        # Target correction is "truth - model" (like a nudging/ML tendency target)
        self.y = (truth - model).astype(np.float32)

        # Simple feature scaling (replace with fit stats from training set!)
        x_mean = self.x.mean(axis=(0, 2, 3), keepdims=True)
        x_std  = self.x.std(axis=(0, 2, 3), keepdims=True) + 1e-6
        self.x = (self.x - x_mean) / x_std

        self.x = torch.from_numpy(self.x)
        self.y = torch.from_numpy(self.y)

    def __len__(self): return self.x.shape[0]
    def __getitem__(self, i): return self.x[i], self.y[i]

# --------------------------
# 2) U-Net building blocks
# --------------------------
class ConvBlock(nn.Module):
    def __init__(self, c_in, c_out, k=3):
        super().__init__()
        p = k // 2
        self.net = nn.Sequential(
            nn.Conv2d(c_in, c_out, k, padding=p),
            nn.GroupNorm( min(32, c_out), c_out),
            nn.GELU(),
            nn.Conv2d(c_out, c_out, k, padding=p),
            nn.GroupNorm( min(32, c_out), c_out),
            nn.GELU(),
        )
    def forward(self, x): return self.net(x)

class UNetSmall(nn.Module):
    def __init__(self, c_in, c_out, width=48, depth=3):
        super().__init__()
        ch = [width * (2**i) for i in range(depth+1)]

        # Encoder
        self.enc = nn.ModuleList()
        self.down = nn.ModuleList()
        cprev = c_in
        for i in range(depth):
            self.enc.append(ConvBlock(cprev, ch[i]))
            self.down.append(nn.Conv2d(ch[i], ch[i], 4, stride=2, padding=1))
            cprev = ch[i]
        self.mid = ConvBlock(ch[depth-1], ch[depth])

        # Decoder
        self.up = nn.ModuleList()
        self.dec = nn.ModuleList()
        for i in reversed(range(depth)):
            self.up.append(nn.ConvTranspose2d(ch[i+1], ch[i], 4, stride=2, padding=1))
            self.dec.append(ConvBlock(ch[i]*2, ch[i]))
        self.out = nn.Conv2d(ch[0], c_out, 1)

    def forward(self, x):
        skips = []
        h = x
        for enc, down in zip(self.enc, self.down):
            h = enc(h)
            skips.append(h)
            h = down(h)
        h = self.mid(h)
        for up, dec in zip(self.up, self.dec):
            h = up(h)
            s = skips.pop()
            # handle slight mismatches from odd shapes
            if h.shape[-2:] != s.shape[-2:]:
                h = nn.functional.interpolate(h, size=s.shape[-2:], mode="bilinear", align_corners=False)
            h = torch.cat([h, s], dim=1)
            h = dec(h)
        return self.out(h)

# --------------------------
# 3) Training loop
# --------------------------
@dataclass
class TrainCfg:
    c_in: int = 8
    c_out: int = 2
    height: int = 64
    width: int = 128
    batch_size: int = 16
    epochs: int = 10
    lr: float = 3e-4
    weight_decay: float = 1e-5
    l1_alpha: float = 0.01       # small L1 to encourage sparsity
    grad_clip: float = 1.0

def train(cfg=TrainCfg()):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    ds = ToyBiasDataset(n=2000, c_in=cfg.c_in, c_out=cfg.c_out,
                        h=cfg.height, w=cfg.width, seed=42)
    n_train = int(0.9 * len(ds))
    tr, va = torch.utils.data.random_split(ds, [n_train, len(ds)-n_train],
                                           generator=torch.Generator().manual_seed(0))
    dl_tr = DataLoader(tr, batch_size=cfg.batch_size, shuffle=True, num_workers=2, pin_memory=True)
    dl_va = DataLoader(va, batch_size=cfg.batch_size, shuffle=False, num_workers=2, pin_memory=True)

    net = UNetSmall(cfg.c_in, cfg.c_out, width=48, depth=3).to(device)
    opt = torch.optim.AdamW(net.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    mse = nn.MSELoss()

    best = math.inf
    for epoch in range(1, cfg.epochs+1):
        net.train()
        tr_loss = 0.0
        for x, y in dl_tr:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            yhat = net(x)
            loss = mse(yhat, y) + cfg.l1_alpha * (yhat.abs().mean())
            loss.backward()
            nn.utils.clip_grad_norm_(net.parameters(), cfg.grad_clip)
            opt.step()
            tr_loss += loss.item() * x.size(0)
        tr_loss /= len(tr)

        net.eval()
        va_loss = 0.0
        with torch.no_grad():
            for x, y in dl_va:
                x, y = x.to(device), y.to(device)
                yhat = net(x)
                va_loss += mse(yhat, y).item() * x.size(0)
        va_loss /= len(va)
        print(f"Epoch {epoch:02d} | train {tr_loss:.4f} | valid {va_loss:.4f}")

        if va_loss < best:
            best = va_loss
            torch.save(net.state_dict(), "unet_bias_best.pt")
    print("Done. Best valid MSE:", best)

if __name__ == "__main__":
    train()



ValueError: num_channels must be divisible by num_groups